# Data Collection and Processing

Tools for collecting training data and monitoring dataset statistics.

In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('.'))))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

import config

%matplotlib inline

## Check Collected Videos

In [ ]:
# List all recorded videos
video_files = list(config.RAW_DATA_DIR.glob("*.mp4"))
print(f"Found {len(video_files)} video clips\n")

if video_files:
    # Parse and display video metadata
    video_info = []
    for video_path in sorted(video_files):
        parts = video_path.stem.split('_')
        if len(parts) >= 4:
            video_info.append({
                'filename': video_path.name,
                'person': parts[0],
                'session': parts[1],
                'state': parts[2],
                'size_mb': video_path.stat().st_size / (1024 * 1024)
            })
    
    df_videos = pd.DataFrame(video_info)
    
    # Summary by person and state
    print("Videos by person and state:")
    summary = df_videos.groupby(['person', 'state']).size().unstack(fill_value=0)
    print(summary)
    print(f"\nTotal size: {df_videos['size_mb'].sum():.1f} MB")
else:
    print("No videos found. Run src/collect_data.py to record clips.")

## Preview Video Frames

In [ ]:
def preview_video(video_path, n_frames=6):
    """Preview frames from a video."""
    cap = cv2.VideoCapture(str(video_path))
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, total_frames-1, n_frames, dtype=int)
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
    cap.release()
    
    # Display frames
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    fig.suptitle(f"Preview: {video_path.name}", fontsize=14)
    
    for i, (ax, frame) in enumerate(zip(axes.flat, frames)):
        ax.imshow(frame)
        ax.set_title(f"Frame {frame_indices[i]}")
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Preview first video if available
if video_files:
    preview_video(video_files[0])

## Check Extracted Frames

In [ ]:
# Count extracted frames
frame_counts = {}
for state, state_dir in [
    ('focused', config.FOCUSED_FRAMES_DIR),
    ('drowsy', config.DROWSY_FRAMES_DIR),
    ('distracted', config.DISTRACTED_FRAMES_DIR)
]:
    frames = list(state_dir.glob("*.jpg"))
    frame_counts[state] = len(frames)

print("Extracted frames by class:")
for state, count in frame_counts.items():
    print(f"  {state.capitalize()}: {count}")

total_frames = sum(frame_counts.values())
print(f"\nTotal frames: {total_frames}")

if total_frames > 0:
    # Visualize class distribution
    plt.figure(figsize=(8, 5))
    plt.bar(frame_counts.keys(), frame_counts.values())
    plt.title("Frame Distribution by Class")
    plt.xlabel("State")
    plt.ylabel("Number of Frames")
    plt.axhline(y=min(frame_counts.values()), color='r', linestyle='--', 
                label=f'Min count: {min(frame_counts.values())}')
    plt.legend()
    plt.show()
else:
    print("No frames extracted yet. Run src/extract_frames.py to process videos.")

## Visualize Sample Frames

In [ ]:
def show_sample_frames(n_per_class=3):
    """Display sample frames from each class."""
    
    fig, axes = plt.subplots(3, n_per_class, figsize=(12, 9))
    fig.suptitle("Sample Frames by Class", fontsize=14)
    
    for i, (state, state_dir) in enumerate([
        ('focused', config.FOCUSED_FRAMES_DIR),
        ('drowsy', config.DROWSY_FRAMES_DIR),
        ('distracted', config.DISTRACTED_FRAMES_DIR)
    ]):
        frames = list(state_dir.glob("*.jpg"))
        
        if frames:
            # Sample random frames
            sample = np.random.choice(frames, min(n_per_class, len(frames)), replace=False)
            
            for j, frame_path in enumerate(sample):
                img = cv2.imread(str(frame_path))
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                ax = axes[i, j] if n_per_class > 1 else axes[i]
                ax.imshow(img_rgb)
                ax.axis('off')
                
                if j == 0:
                    ax.set_title(state.capitalize(), fontweight='bold')
        else:
            for j in range(n_per_class):
                ax = axes[i, j] if n_per_class > 1 else axes[i]
                ax.axis('off')
                ax.text(0.5, 0.5, f'No {state} frames', 
                       ha='center', va='center', transform=ax.transAxes)
    
    plt.tight_layout()
    plt.show()

if total_frames > 0:
    show_sample_frames()

## Session Distribution Analysis

In [ ]:
def analyze_sessions():
    """Analyze distribution of frames across sessions."""
    
    session_data = []
    
    for state, state_dir in [
        ('focused', config.FOCUSED_FRAMES_DIR),
        ('drowsy', config.DROWSY_FRAMES_DIR),
        ('distracted', config.DISTRACTED_FRAMES_DIR)
    ]:
        frames = list(state_dir.glob("*.jpg"))
        
        for frame_path in frames:
            # Parse filename: person_session_state_frame.jpg
            parts = frame_path.stem.split('_')
            if len(parts) >= 4:
                session_data.append({
                    'person': parts[0],
                    'session': parts[1],
                    'state': state,
                    'frame': frame_path.name
                })
    
    if session_data:
        df = pd.DataFrame(session_data)
        
        print("Frames by person and session:")
        session_summary = df.groupby(['person', 'session', 'state']).size().unstack(fill_value=0)
        print(session_summary)
        
        print("\nUnique sessions:", df.groupby(['person', 'session']).ngroups)
        print("\nThis session information will be used for train/test split")
        print("to prevent data leakage between similar frames.")
    else:
        print("No frame data to analyze")

if total_frames > 0:
    analyze_sessions()